In [1]:
from os import environ, makedirs, path

import pandas as pd
from dotenv import load_dotenv
from unidecode import unidecode

In [2]:
load_dotenv()

True

# Atualização de 2026 - Mortalidade Materna e Mortalidade Infantil

Para o ano de 2026, o GT Saúde decidiu adicionar dois painéis com dados específicos de mortalidade, um para mortalidade materna e outro para mortalidade infantil. Esses temas já haviam sido sugeridos pelos especialistas que participaram em 2025, e também foram trabalhados pela auditoria do TCMSP, resultando inclusive em um evento em parceria com o OPP no ano de 2026.

Agora, precisamos adicionar esses dados padronizados no formato adequado ao Qlik Sense para que sejam incorporados ao DataSP.

Os dados são coeficientes semelhantes ao de mortalidade geral, mas com recortes específicos. De todo modo, tratam-se de coeficientes ponderados. No caso de mortalidade materna, temos o seguinte coeficiente:

$$ \text{Razão de mortalidade materna} = \frac{\text{Número de óbitos de mulheres por causas e condições consideradas de morte materna}}{\text{Número de nascidos vivos}} \times 100.000 $$

E, no caso de mortalidade infantil, temos:

$$ \text{Taxa de mortalidade infantil} = \frac{\text{Número de óbitos menores de 1 ano de idade}}{\text{Número de nascidos vivos}} \times 1.000 $$

Para que possamos recalcular esses indicadores de acordo com a seleção de Subprefeituras feito dinamicamente pelo usuário, precisamos obter o numerador e o denomidador deles e agregá-los dinamicamente de acordo com a seleção.

Para isso, existem duas alternativas possíveis. Uma delas é obter a razão (ou taxa) de mortalidade já calculada para cada Subprefeitura e estimar o número de óbitos com base no número de nascidos vivos. Outra forma seria obter os componentes da razão (taxa) e calculá-lo para cada Subprefeitura. A primeira maneira é preferida, porque garante que os indicadores reflitam exatamente os valores disponíveis em bases de dados oficiais. Entretanto, esses valores existem apenas para o ano de 2024. Para o ano de 2025, estão disponíveis apenas dados provisórios de mortalidade geral, de modo que o cálculo da razão (taxa) é possível de modo provisório, ainda que não esteja disponível em bases oficiais.

Portanto, as duas maneiras serão utilizadas: para o ano de 2024, serão utilizados os valores diponíveis em bases oficiais e o número de óbitos será estimado com base no número de nascidos vivos; para o ano de 2025, a razão (taxa) será calculada com base nos dados de mortalidade geral e nascidos vivos.

# Extração dos dados

## Mortalidade Materna

Esta razão será obtida do ObservaSampa, mas o download está apresentando um erro no momento. Os dados foram extraídos e corrigidos manualmente e estão disponíveis na pasta de dados do projeto

### 2024

In [3]:
data_input_dir = path.join('data', 'input', 'saude')

In [4]:
file_name_materna = 'mortalidade_materna_2024.csv'
df_materna_24 = pd.read_csv(path.join(data_input_dir, file_name_materna), sep=';', encoding='latin1', decimal=',')
df_materna_24

,região,nivel_regional,indicador,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,São Paulo,Município,03.01.01 Razão de mortalidade materna (por 100...,59.5049,49.4625,49.1477,62.5654,58.4294,56.7917,59.0706,47.7766,50.4430,56.4131,65.0961,35.5896,29.5247,25.6544
1,Guaianases,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,120.6030,39.8248,138.5864,153.6393,118.8119,59.8563,155.6723,60.9632,63.5997,45.1365,0.0000,0.0000,79.7448,56.9638
2,Ipiranga,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,15.1883,76.4292,15.2999,30.1114,44.6163,77.9180,15.7978,15.5933,16.7898,35.6888,58.3771,0.0000,41.0256,21.5936
3,Pinheiros,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,96.3391,32.4044,33.0251,32.4570,0.0000,34.3525,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,Pirituba-Jaraguá,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,42.3430,57.6037,29.5639,43.3839,72.1605,77.1724,76.2544,31.5657,64.9351,88.9363,55.5042,56.9801,37.6152,20.3707
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,Vila Guilherme,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,446.4286,0.0000,153.6098,0.0000
125,Vila Matilde,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,203.6660,99.2063,104.4932,109.0513
126,Vila Prudente,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,90.4977,0.0000,0.0000,0.0000
127,Penha,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,79.4913


### 2025

Para 2025, obteremos os dados de mortalidade geral filtrando para mulheres com causas relacionadas ao Capítulo XV. Gravidez parto e puerpério do CID10. Esses dados também são extraídos manualmente, desta vez do TabNet, e adicionados a pasta de dados deste projeto.

In [5]:
file_name_obitos_materna_25 = 'A002518192_29_143_95.csv'
df_obitos_materna_25 = pd.read_csv(path.join(data_input_dir, file_name_obitos_materna_25),
                                   sep=';', encoding='latin1', decimal=',', skiprows=5)
df_obitos_materna_25

,Subprefeitura residência,Óbitos_Residentes_MSP
0,Aricanduva/Formosa/Carrão,2
1,Butantã,1
2,Campo Limpo,1
3,Capela do Socorro,3
4,Casa Verde/Cachoeirinha,-
5,Cidade Ademar,2
6,Cidade Tiradentes,-
7,Ermelino Matarazzo,-
8,Freguesia/Brasilândia,-
9,Guaianases,1


Vamos remover as linhas finais com informações complementares sobre os dados

In [6]:
df_obitos_materna_25 = df_obitos_materna_25[:-6]
df_obitos_materna_25

,Subprefeitura residência,Óbitos_Residentes_MSP
0,Aricanduva/Formosa/Carrão,2
1,Butantã,1
2,Campo Limpo,1
3,Capela do Socorro,3
4,Casa Verde/Cachoeirinha,-
5,Cidade Ademar,2
6,Cidade Tiradentes,-
7,Ermelino Matarazzo,-
8,Freguesia/Brasilândia,-
9,Guaianases,1


## Mortalidade Infantil

### 2024

Neste caso, tanto a taxa, quanto os dados provisórios de óbitos serão obtidos do TabNet. Vamos começar pela taxa de mortalidade infantil.

In [7]:
file_name_infantil_24 = 'A003317192_29_141_165.csv'
df_infantil_24 = pd.read_csv(path.join(data_input_dir, file_name_infantil_24),
                                         sep=';', encoding='latin1', decimal=',', skiprows=3)
df_infantil_24

,Subprefeitura resid,Coef_mort_infantil
0,Aricanduva/Formosa/Carrão,11.94
1,Butantã,8.41
2,Campo Limpo,10.25
3,Capela do Socorro,12.97
4,Casa Verde/Cachoeirinha,11.43
5,Cidade Ademar,11.77
6,Cidade Tiradentes,9.01
7,Ermelino Matarazzo,8.26
8,Freguesia/Brasilândia,15.18
9,Guaianases,13.96


Novamente, vamos remover as linhas de totalização e informações adicionais.

In [8]:
df_infantil_24 = df_infantil_24[:-8]
df_infantil_24

,Subprefeitura resid,Coef_mort_infantil
0,Aricanduva/Formosa/Carrão,11.94
1,Butantã,8.41
2,Campo Limpo,10.25
3,Capela do Socorro,12.97
4,Casa Verde/Cachoeirinha,11.43
5,Cidade Ademar,11.77
6,Cidade Tiradentes,9.01
7,Ermelino Matarazzo,8.26
8,Freguesia/Brasilândia,15.18
9,Guaianases,13.96


### 2025

In [9]:
file_name_obitos_infantil_25 = 'A003858192_29_141_165.csv'
df_obitos_infantil_25 = pd.read_csv(path.join(data_input_dir, file_name_obitos_infantil_25),
                                    sep=';', encoding='latin1', decimal=',', skiprows=4)
df_obitos_infantil_25 = df_obitos_infantil_25[:-6]
df_obitos_infantil_25

,Subprefeitura residência,Óbitos_Residentes_MSP
0,Aricanduva/Formosa/Carrão,18.0
1,Butantã,41.0
2,Campo Limpo,66.0
3,Capela do Socorro,79.0
4,Casa Verde/Cachoeirinha,37.0
5,Cidade Ademar,57.0
6,Cidade Tiradentes,32.0
7,Ermelino Matarazzo,14.0
8,Freguesia/Brasilândia,68.0
9,Guaianases,37.0


## Nascidos Vivos

Os dados de nascidos vivos também são obtidos do TabNet.

In [10]:
file_name_nv = 'A004130192_29_141_165.csv'
df_nv = pd.read_csv(path.join(data_input_dir, file_name_nv),
                    sep=';', encoding='latin1', decimal=',', skiprows=3)
df_nv = df_nv[:-3]
df_nv

,SubPrefeitura residência,2024,2025,Total
0,Aricanduva/Formosa/Carrão,2177.0,2302.0,4479.0
1,Butantã,4399.0,4639.0,9038.0
2,Campo Limpo,7417.0,7368.0,14785.0
3,Capela do Socorro,7092.0,7121.0,14213.0
4,Casa Verde/Cachoeirinha,3236.0,3428.0,6664.0
5,Cidade Ademar,4673.0,4819.0,9492.0
6,Cidade Tiradentes,2552.0,2530.0,5082.0
7,Ermelino Matarazzo,2057.0,2121.0,4178.0
8,Freguesia/Brasilândia,4215.0,4293.0,8508.0
9,Guaianases,3511.0,3400.0,6911.0


Nesse caso, temos também que excluir a coluna de totalização.

In [11]:
df_nv = df_nv.drop(columns=['Total'])
df_nv

,SubPrefeitura residência,2024,2025
0,Aricanduva/Formosa/Carrão,2177.0,2302.0
1,Butantã,4399.0,4639.0
2,Campo Limpo,7417.0,7368.0
3,Capela do Socorro,7092.0,7121.0
4,Casa Verde/Cachoeirinha,3236.0,3428.0
5,Cidade Ademar,4673.0,4819.0
6,Cidade Tiradentes,2552.0,2530.0
7,Ermelino Matarazzo,2057.0,2121.0
8,Freguesia/Brasilândia,4215.0,4293.0
9,Guaianases,3511.0,3400.0


## Subprefeituras

Para que possamos unir os dados aqui e no Qlik precisamos padronizar o nome das Subprefeituras para utilizar como chave. Para isso, vamos carregar o arquivo de subprefeituras utilizado no Qlik.

In [12]:
url_subs = environ.get('CSV_SUBPREFEITURAS_QLIK')
df_subs = pd.read_csv(url_subs)
df_subs

,sub.CODIGO,sub.NOME,sub.POLIGONO,total_de_pessoas
0,1,PERUS,"[[[[-46.7082012617476,-23.4165818252815],[-46....",163076
1,2,PIRITUBA-JARAGUA,"[[[[-46.7753669019781,-23.4628032705496],[-46....",480225
2,3,FREGUESIA-BRASILANDIA,"[[[[-46.6910223492151,-23.5088410763715],[-46....",380513
3,4,CASA VERDE-CACHOEIRINHA,"[[[[-46.673577305455,-23.479858692024],[-46.67...",306275
4,5,SANTANA-TUCURUVI,"[[[[-46.6251942372966,-23.5193645522374],[-46....",318913
5,6,JACANA-TREMEMBE,"[[[[-46.6138088697146,-23.4620183373337],[-46....",283892
6,7,VILA MARIA-VILA GUILHERME,"[[[[-46.5985584306269,-23.4916579527685],[-46....",276069
7,8,LAPA,"[[[[-46.7475634318888,-23.5587659797062],[-46....",338347
8,9,SE,"[[[[-46.6634017492882,-23.5366395157441],[-46....",423536
9,10,BUTANTA,"[[[[-46.7579053376809,-23.5510466758162],[-46....",468522


# Transformação dos dados

## Padronizando os nomes de subprefeituras

In [13]:
df_subs_qlik = df_subs['sub.NOME'].sort_values().tolist()
df_subs_qlik

['ARICANDUVA-FORMOSA-CARRAO',
 'BUTANTA',
 'CAMPO LIMPO',
 'CAPELA DO SOCORRO',
 'CASA VERDE-CACHOEIRINHA',
 'CIDADE ADEMAR',
 'CIDADE TIRADENTES',
 'ERMELINO MATARAZZO',
 'FREGUESIA-BRASILANDIA',
 'GUAIANASES',
 'IPIRANGA',
 'ITAIM PAULISTA',
 'ITAQUERA',
 'JABAQUARA',
 'JACANA-TREMEMBE',
 'LAPA',
 'M BOI MIRIM',
 'MOOCA',
 'PARELHEIROS',
 'PENHA',
 'PERUS',
 'PINHEIROS',
 'PIRITUBA-JARAGUA',
 'SANTANA-TUCURUVI',
 'SANTO AMARO',
 'SAO MATEUS',
 'SAO MIGUEL',
 'SAPOPEMBA',
 'SE',
 'VILA MARIA-VILA GUILHERME',
 'VILA MARIANA',
 'VILA PRUDENTE']

### Mortalidade Materna

In [14]:
subs_materna_24 = (
    df_materna_24[df_materna_24['nivel_regional'] == 'Subprefeitura']
    .assign(Subprefeitura=df_materna_24['região'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['região']
    .tolist()
)
subs_materna_24

['Aricanduva-Formosa-Carrão',
 'Butantã',
 'Campo Limpo',
 'Capela Do Socorro',
 'Casa Verde-Cachoeirinha',
 'Cidade Ademar',
 'Cidade Tiradentes',
 'Ermelino Matarazzo',
 'Freguesia-Brasilândia',
 'Guaianases',
 'Ipiranga',
 'Itaim Paulista',
 'Itaquera',
 'Jabaquara',
 'Jaçanã-Tremembé',
 'Lapa',
 "M'Boi Mirim",
 'Mooca',
 'Parelheiros',
 'Penha',
 'Perus',
 'Pinheiros',
 'Pirituba-Jaraguá',
 'Santana-Tucuruvi',
 'Santo Amaro',
 'São Mateus',
 'São Miguel',
 'Sapopemba',
 'Sé',
 'Vila Maria-Vila Guilherme',
 'Vila Mariana',
 'Vila Prudente']

In [15]:
map_materna_24 = dict(zip(subs_materna_24, df_subs_qlik))
map_materna_24

{'Aricanduva-Formosa-Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
 'Butantã': 'BUTANTA',
 'Campo Limpo': 'CAMPO LIMPO',
 'Capela Do Socorro': 'CAPELA DO SOCORRO',
 'Casa Verde-Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Cidade Ademar': 'CIDADE ADEMAR',
 'Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
 'Freguesia-Brasilândia': 'FREGUESIA-BRASILANDIA',
 'Guaianases': 'GUAIANASES',
 'Ipiranga': 'IPIRANGA',
 'Itaim Paulista': 'ITAIM PAULISTA',
 'Itaquera': 'ITAQUERA',
 'Jabaquara': 'JABAQUARA',
 'Jaçanã-Tremembé': 'JACANA-TREMEMBE',
 'Lapa': 'LAPA',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Mooca': 'MOOCA',
 'Parelheiros': 'PARELHEIROS',
 'Penha': 'PENHA',
 'Perus': 'PERUS',
 'Pinheiros': 'PINHEIROS',
 'Pirituba-Jaraguá': 'PIRITUBA-JARAGUA',
 'Santana-Tucuruvi': 'SANTANA-TUCURUVI',
 'Santo Amaro': 'SANTO AMARO',
 'São Mateus': 'SAO MATEUS',
 'São Miguel': 'SAO MIGUEL',
 'Sapopemba': 'SAPOPEMBA',
 'Sé': 'SE',
 'Vila Maria-Vila Guilherme': 'VILA MARIA-VILA GUILHERME',


In [16]:
df_materna_24['sub.NOME'] = df_materna_24['região'].map(map_materna_24)
df_materna_24

,região,nivel_regional,indicador,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,sub.NOME
0,São Paulo,Município,03.01.01 Razão de mortalidade materna (por 100...,59.5049,49.4625,49.1477,62.5654,58.4294,56.7917,59.0706,47.7766,50.4430,56.4131,65.0961,35.5896,29.5247,25.6544,NaN
1,Guaianases,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,120.6030,39.8248,138.5864,153.6393,118.8119,59.8563,155.6723,60.9632,63.5997,45.1365,0.0000,0.0000,79.7448,56.9638,GUAIANASES
2,Ipiranga,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,15.1883,76.4292,15.2999,30.1114,44.6163,77.9180,15.7978,15.5933,16.7898,35.6888,58.3771,0.0000,41.0256,21.5936,IPIRANGA
3,Pinheiros,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,96.3391,32.4044,33.0251,32.4570,0.0000,34.3525,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,PINHEIROS
4,Pirituba-Jaraguá,Subprefeitura,03.01.01 Razão de mortalidade materna (por 100...,42.3430,57.6037,29.5639,43.3839,72.1605,77.1724,76.2544,31.5657,64.9351,88.9363,55.5042,56.9801,37.6152,20.3707,PIRITUBA-JARAGUA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,Vila Guilherme,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,446.4286,0.0000,153.6098,0.0000,NaN
125,Vila Matilde,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,203.6660,99.2063,104.4932,109.0513,NaN
126,Vila Prudente,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,90.4977,0.0000,0.0000,0.0000,VILA PRUDENTE
127,Penha,Distrito,03.01.01 Razão de mortalidade materna (por 100...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,79.4913,PENHA


In [17]:
subs_materna_25 = (
    df_obitos_materna_25
    .assign(Subprefeitura=df_obitos_materna_25['Subprefeitura residência'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['Subprefeitura residência']
    .tolist()
)
subs_materna_25.remove('Ignorado')
subs_materna_25

['Aricanduva/Formosa/Carrão',
 'Butantã',
 'Campo Limpo',
 'Capela do Socorro',
 'Casa Verde/Cachoeirinha',
 'Cidade Ademar',
 'Cidade Tiradentes',
 'Ermelino Matarazzo',
 'Freguesia/Brasilândia',
 'Guaianases',
 'Ipiranga',
 'Itaim Paulista',
 'Itaquera',
 'Jabaquara',
 'Jaçanã/Tremembé',
 'Lapa',
 "M'Boi Mirim",
 'Mooca',
 'Parelheiros',
 'Penha',
 'Perus',
 'Pinheiros',
 'Pirituba',
 'Santana/Tucuruvi',
 'Santo Amaro',
 'São Mateus',
 'São Miguel',
 'Sapopemba',
 'Sé',
 'Vila Maria/Vila Guilherme',
 'Vila Mariana',
 'Vila Prudente']

In [18]:
map_materna_25 = dict(zip(subs_materna_25, df_subs_qlik))
map_materna_25

{'Aricanduva/Formosa/Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
 'Butantã': 'BUTANTA',
 'Campo Limpo': 'CAMPO LIMPO',
 'Capela do Socorro': 'CAPELA DO SOCORRO',
 'Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Cidade Ademar': 'CIDADE ADEMAR',
 'Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
 'Freguesia/Brasilândia': 'FREGUESIA-BRASILANDIA',
 'Guaianases': 'GUAIANASES',
 'Ipiranga': 'IPIRANGA',
 'Itaim Paulista': 'ITAIM PAULISTA',
 'Itaquera': 'ITAQUERA',
 'Jabaquara': 'JABAQUARA',
 'Jaçanã/Tremembé': 'JACANA-TREMEMBE',
 'Lapa': 'LAPA',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Mooca': 'MOOCA',
 'Parelheiros': 'PARELHEIROS',
 'Penha': 'PENHA',
 'Perus': 'PERUS',
 'Pinheiros': 'PINHEIROS',
 'Pirituba': 'PIRITUBA-JARAGUA',
 'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
 'Santo Amaro': 'SANTO AMARO',
 'São Mateus': 'SAO MATEUS',
 'São Miguel': 'SAO MIGUEL',
 'Sapopemba': 'SAPOPEMBA',
 'Sé': 'SE',
 'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
 'Vila M

In [19]:
df_obitos_materna_25.loc[:, 'sub.NOME'] = df_obitos_materna_25['Subprefeitura residência'].map(map_materna_25)
df_obitos_materna_25

<positron-console-cell-19>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


,Subprefeitura residência,Óbitos_Residentes_MSP,sub.NOME
0,Aricanduva/Formosa/Carrão,2,ARICANDUVA-FORMOSA-CARRAO
1,Butantã,1,BUTANTA
2,Campo Limpo,1,CAMPO LIMPO
3,Capela do Socorro,3,CAPELA DO SOCORRO
4,Casa Verde/Cachoeirinha,-,CASA VERDE-CACHOEIRINHA
5,Cidade Ademar,2,CIDADE ADEMAR
6,Cidade Tiradentes,-,CIDADE TIRADENTES
7,Ermelino Matarazzo,-,ERMELINO MATARAZZO
8,Freguesia/Brasilândia,-,FREGUESIA-BRASILANDIA
9,Guaianases,1,GUAIANASES


### Mortalidade Infantil

In [20]:
subs_infantil_24 = (
    df_infantil_24
    .assign(Subprefeitura=df_infantil_24['Subprefeitura resid'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['Subprefeitura resid']
    .tolist()
)
subs_infantil_24.remove('Ignorado')
subs_infantil_24

['Aricanduva/Formosa/Carrão',
 'Butantã',
 'Campo Limpo',
 'Capela do Socorro',
 'Casa Verde/Cachoeirinha',
 'Cidade Ademar',
 'Cidade Tiradentes',
 'Ermelino Matarazzo',
 'Freguesia/Brasilândia',
 'Guaianases',
 'Ipiranga',
 'Itaim Paulista',
 'Itaquera',
 'Jabaquara',
 'Jaçanã/Tremembé',
 'Lapa',
 "M'Boi Mirim",
 'Mooca',
 'Parelheiros',
 'Penha',
 'Perus',
 'Pinheiros',
 'Pirituba',
 'Santana/Tucuruvi',
 'Santo Amaro',
 'São Mateus',
 'São Miguel',
 'Sapopemba',
 'Sé',
 'Vila Maria/Vila Guilherme',
 'Vila Mariana',
 'Vila Prudente']

In [21]:
map_infantil_24 = dict(zip(subs_infantil_24, df_subs_qlik))
map_infantil_24

{'Aricanduva/Formosa/Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
 'Butantã': 'BUTANTA',
 'Campo Limpo': 'CAMPO LIMPO',
 'Capela do Socorro': 'CAPELA DO SOCORRO',
 'Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Cidade Ademar': 'CIDADE ADEMAR',
 'Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
 'Freguesia/Brasilândia': 'FREGUESIA-BRASILANDIA',
 'Guaianases': 'GUAIANASES',
 'Ipiranga': 'IPIRANGA',
 'Itaim Paulista': 'ITAIM PAULISTA',
 'Itaquera': 'ITAQUERA',
 'Jabaquara': 'JABAQUARA',
 'Jaçanã/Tremembé': 'JACANA-TREMEMBE',
 'Lapa': 'LAPA',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Mooca': 'MOOCA',
 'Parelheiros': 'PARELHEIROS',
 'Penha': 'PENHA',
 'Perus': 'PERUS',
 'Pinheiros': 'PINHEIROS',
 'Pirituba': 'PIRITUBA-JARAGUA',
 'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
 'Santo Amaro': 'SANTO AMARO',
 'São Mateus': 'SAO MATEUS',
 'São Miguel': 'SAO MIGUEL',
 'Sapopemba': 'SAPOPEMBA',
 'Sé': 'SE',
 'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
 'Vila M

In [22]:
df_infantil_24.loc[:, 'sub.NOME'] = df_infantil_24.loc[:, 'Subprefeitura resid'].map(map_infantil_24)
df_infantil_24

<positron-console-cell-22>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


,Subprefeitura resid,Coef_mort_infantil,sub.NOME
0,Aricanduva/Formosa/Carrão,11.94,ARICANDUVA-FORMOSA-CARRAO
1,Butantã,8.41,BUTANTA
2,Campo Limpo,10.25,CAMPO LIMPO
3,Capela do Socorro,12.97,CAPELA DO SOCORRO
4,Casa Verde/Cachoeirinha,11.43,CASA VERDE-CACHOEIRINHA
5,Cidade Ademar,11.77,CIDADE ADEMAR
6,Cidade Tiradentes,9.01,CIDADE TIRADENTES
7,Ermelino Matarazzo,8.26,ERMELINO MATARAZZO
8,Freguesia/Brasilândia,15.18,FREGUESIA-BRASILANDIA
9,Guaianases,13.96,GUAIANASES


In [23]:
subs_infantil_25 = (
    df_obitos_infantil_25
    .assign(Subprefeitura=df_obitos_infantil_25['Subprefeitura residência'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['Subprefeitura residência']
    .tolist()
)
subs_infantil_25.remove('Ignorado')
subs_infantil_25

['Aricanduva/Formosa/Carrão',
 'Butantã',
 'Campo Limpo',
 'Capela do Socorro',
 'Casa Verde/Cachoeirinha',
 'Cidade Ademar',
 'Cidade Tiradentes',
 'Ermelino Matarazzo',
 'Freguesia/Brasilândia',
 'Guaianases',
 'Ipiranga',
 'Itaim Paulista',
 'Itaquera',
 'Jabaquara',
 'Jaçanã/Tremembé',
 'Lapa',
 "M'Boi Mirim",
 'Moóca',
 'Parelheiros',
 'Penha',
 'Perus',
 'Pinheiros',
 'Pirituba',
 'Santana/Tucuruvi',
 'Santo Amaro',
 'São Mateus',
 'São Miguel',
 'Sapopemba',
 'Sé',
 'Vila Maria/Vila Guilherme',
 'Vila Mariana',
 'Vila Prudente']

In [24]:
map_infantil_25 = dict(zip(subs_infantil_25, df_subs_qlik))
map_infantil_25

{'Aricanduva/Formosa/Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
 'Butantã': 'BUTANTA',
 'Campo Limpo': 'CAMPO LIMPO',
 'Capela do Socorro': 'CAPELA DO SOCORRO',
 'Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Cidade Ademar': 'CIDADE ADEMAR',
 'Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
 'Freguesia/Brasilândia': 'FREGUESIA-BRASILANDIA',
 'Guaianases': 'GUAIANASES',
 'Ipiranga': 'IPIRANGA',
 'Itaim Paulista': 'ITAIM PAULISTA',
 'Itaquera': 'ITAQUERA',
 'Jabaquara': 'JABAQUARA',
 'Jaçanã/Tremembé': 'JACANA-TREMEMBE',
 'Lapa': 'LAPA',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Moóca': 'MOOCA',
 'Parelheiros': 'PARELHEIROS',
 'Penha': 'PENHA',
 'Perus': 'PERUS',
 'Pinheiros': 'PINHEIROS',
 'Pirituba': 'PIRITUBA-JARAGUA',
 'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
 'Santo Amaro': 'SANTO AMARO',
 'São Mateus': 'SAO MATEUS',
 'São Miguel': 'SAO MIGUEL',
 'Sapopemba': 'SAPOPEMBA',
 'Sé': 'SE',
 'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
 'Vila M

In [25]:
df_obitos_infantil_25.loc[:, 'sub.NOME'] = df_obitos_infantil_25.loc[:, 'Subprefeitura residência'].map(map_infantil_25)
df_obitos_infantil_25

,Subprefeitura residência,Óbitos_Residentes_MSP,sub.NOME
0,Aricanduva/Formosa/Carrão,18.0,ARICANDUVA-FORMOSA-CARRAO
1,Butantã,41.0,BUTANTA
2,Campo Limpo,66.0,CAMPO LIMPO
3,Capela do Socorro,79.0,CAPELA DO SOCORRO
4,Casa Verde/Cachoeirinha,37.0,CASA VERDE-CACHOEIRINHA
5,Cidade Ademar,57.0,CIDADE ADEMAR
6,Cidade Tiradentes,32.0,CIDADE TIRADENTES
7,Ermelino Matarazzo,14.0,ERMELINO MATARAZZO
8,Freguesia/Brasilândia,68.0,FREGUESIA-BRASILANDIA
9,Guaianases,37.0,GUAIANASES


### Nascidos Vivos

In [26]:
subs_nv = (
    df_nv
    .assign(Subprefeitura=df_nv['SubPrefeitura residência'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['SubPrefeitura residência']
    .tolist()
)
subs_nv.remove('Ignorado')
subs_nv

['Aricanduva/Formosa/Carrão',
 'Butantã',
 'Campo Limpo',
 'Capela do Socorro',
 'Casa Verde/Cachoeirinha',
 'Cidade Ademar',
 'Cidade Tiradentes',
 'Ermelino Matarazzo',
 'Freguesia/Brasilândia',
 'Guaianases',
 'Ipiranga',
 'Itaim Paulista',
 'Itaquera',
 'Jabaquara',
 'Jaçanã/Tremembé',
 'Lapa',
 "M'Boi Mirim",
 'Mooca',
 'Parelheiros',
 'Penha',
 'Perus',
 'Pinheiros',
 'Pirituba',
 'Santana/Tucuruvi',
 'Santo Amaro',
 'São Mateus',
 'São Miguel',
 'Sapopemba',
 'Sé',
 'Vila Maria/Vila Guilherme',
 'Vila Mariana',
 'Vila Prudente']

In [27]:
map_nv = dict(zip(subs_nv, df_subs_qlik))
map_nv

{'Aricanduva/Formosa/Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
 'Butantã': 'BUTANTA',
 'Campo Limpo': 'CAMPO LIMPO',
 'Capela do Socorro': 'CAPELA DO SOCORRO',
 'Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Cidade Ademar': 'CIDADE ADEMAR',
 'Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
 'Freguesia/Brasilândia': 'FREGUESIA-BRASILANDIA',
 'Guaianases': 'GUAIANASES',
 'Ipiranga': 'IPIRANGA',
 'Itaim Paulista': 'ITAIM PAULISTA',
 'Itaquera': 'ITAQUERA',
 'Jabaquara': 'JABAQUARA',
 'Jaçanã/Tremembé': 'JACANA-TREMEMBE',
 'Lapa': 'LAPA',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Mooca': 'MOOCA',
 'Parelheiros': 'PARELHEIROS',
 'Penha': 'PENHA',
 'Perus': 'PERUS',
 'Pinheiros': 'PINHEIROS',
 'Pirituba': 'PIRITUBA-JARAGUA',
 'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
 'Santo Amaro': 'SANTO AMARO',
 'São Mateus': 'SAO MATEUS',
 'São Miguel': 'SAO MIGUEL',
 'Sapopemba': 'SAPOPEMBA',
 'Sé': 'SE',
 'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
 'Vila M

In [28]:
df_nv.loc[:, 'sub.NOME'] = df_nv.loc[:, 'SubPrefeitura residência'].map(map_nv)
df_nv

,SubPrefeitura residência,2024,2025,sub.NOME
0,Aricanduva/Formosa/Carrão,2177.0,2302.0,ARICANDUVA-FORMOSA-CARRAO
1,Butantã,4399.0,4639.0,BUTANTA
2,Campo Limpo,7417.0,7368.0,CAMPO LIMPO
3,Capela do Socorro,7092.0,7121.0,CAPELA DO SOCORRO
4,Casa Verde/Cachoeirinha,3236.0,3428.0,CASA VERDE-CACHOEIRINHA
5,Cidade Ademar,4673.0,4819.0,CIDADE ADEMAR
6,Cidade Tiradentes,2552.0,2530.0,CIDADE TIRADENTES
7,Ermelino Matarazzo,2057.0,2121.0,ERMELINO MATARAZZO
8,Freguesia/Brasilândia,4215.0,4293.0,FREGUESIA-BRASILANDIA
9,Guaianases,3511.0,3400.0,GUAIANASES


## Mortalidade Materna

Primeiro, vamos manter apenas o nível de subprefeitura e ano de 2024 para os dados vindos do ObservaSampa.

In [29]:
df_materna_24 = df_materna_24[df_materna_24['nivel_regional'] == 'Subprefeitura']
df_materna_24 = df_materna_24[['região', 'sub.NOME', '2024']]
df_materna_24 = df_materna_24.rename(columns={'região': 'Subprefeitura'})
df_materna_24

,Subprefeitura,sub.NOME,2024
1,Guaianases,GUAIANASES,56.9638
2,Ipiranga,IPIRANGA,21.5936
3,Pinheiros,PINHEIROS,0.0000
4,Pirituba-Jaraguá,PIRITUBA-JARAGUA,20.3707
5,Cidade Ademar,CIDADE ADEMAR,21.4087
6,Penha,PENHA,64.8508
7,Perus,PERUS,0.0000
8,Santana-Tucuruvi,SANTANA-TUCURUVI,0.0000
9,Jaçanã-Tremembé,JACANA-TREMEMBE,56.9638
10,Vila Mariana,VILA MARIANA,0.0000


Agora, vamos transformar o ano em uma coluna.

In [30]:
df_materna_24 = pd.melt(df_materna_24,
                        id_vars=['Subprefeitura', 'sub.NOME'],
                        var_name='Ano',
                        value_name='Razão de mortalidade materna')
df_materna_24

,Subprefeitura,sub.NOME,Ano,Razão de mortalidade materna
0,Guaianases,GUAIANASES,2024,56.9638
1,Ipiranga,IPIRANGA,2024,21.5936
2,Pinheiros,PINHEIROS,2024,0.0000
3,Pirituba-Jaraguá,PIRITUBA-JARAGUA,2024,20.3707
4,Cidade Ademar,CIDADE ADEMAR,2024,21.4087
5,Penha,PENHA,2024,64.8508
6,Perus,PERUS,2024,0.0000
7,Santana-Tucuruvi,SANTANA-TUCURUVI,2024,0.0000
8,Jaçanã-Tremembé,JACANA-TREMEMBE,2024,56.9638
9,Vila Mariana,VILA MARIANA,2024,0.0000


Agora, vamos adicionar a coluna de nascidos vivos para poder calcular o coeficiente.

In [31]:
df_nv  = pd.melt(df_nv,
                 id_vars=['SubPrefeitura residência', 'sub.NOME'],
                 var_name='Ano',
                 value_name='Número de nascidos vivos')
df_nv

,SubPrefeitura residência,sub.NOME,Ano,Número de nascidos vivos
0,Aricanduva/Formosa/Carrão,ARICANDUVA-FORMOSA-CARRAO,2024,2177.0
1,Butantã,BUTANTA,2024,4399.0
2,Campo Limpo,CAMPO LIMPO,2024,7417.0
3,Capela do Socorro,CAPELA DO SOCORRO,2024,7092.0
4,Casa Verde/Cachoeirinha,CASA VERDE-CACHOEIRINHA,2024,3236.0
...,...,...,...,...
61,Sé,SE,2025,3839.0
62,Vila Maria/Vila Guilherme,VILA MARIA-VILA GUILHERME,2025,3700.0
63,Vila Mariana,VILA MARIANA,2025,2721.0
64,Vila Prudente,VILA PRUDENTE,2025,2237.0


In [32]:
df_materna = df_materna_24.merge(
    df_nv[['sub.NOME', 'Ano', 'Número de nascidos vivos']],
    how='left',
    left_on=['sub.NOME', 'Ano'],
    right_on=['sub.NOME', 'Ano']
)
df_materna

,Subprefeitura,sub.NOME,Ano,Razão de mortalidade materna,Número de nascidos vivos
0,Guaianases,GUAIANASES,2024,56.9638,3511.0
1,Ipiranga,IPIRANGA,2024,21.5936,4633.0
2,Pinheiros,PINHEIROS,2024,0.0000,2640.0
3,Pirituba-Jaraguá,PIRITUBA-JARAGUA,2024,20.3707,4935.0
4,Cidade Ademar,CIDADE ADEMAR,2024,21.4087,4673.0
5,Penha,PENHA,2024,64.8508,4628.0
6,Perus,PERUS,2024,0.0000,1855.0
7,Santana-Tucuruvi,SANTANA-TUCURUVI,2024,0.0000,2594.0
8,Jaçanã-Tremembé,JACANA-TREMEMBE,2024,56.9638,3512.0
9,Vila Mariana,VILA MARIANA,2024,0.0000,2663.0


In [33]:
df_materna['Óbitos_Residentes_MSP'] = df_materna['Razão de mortalidade materna'] * df_materna['Número de nascidos vivos'] / 100000
df_materna = df_materna.sort_values(['sub.NOME', 'Ano']).reset_index(drop=True)
df_materna

,Subprefeitura,sub.NOME,Ano,Razão de mortalidade materna,Número de nascidos vivos,Óbitos_Residentes_MSP
0,Aricanduva-Formosa-Carrão,ARICANDUVA-FORMOSA-CARRAO,2024,0.0000,2177.0,0.000000
1,Butantã,BUTANTA,2024,22.7687,4399.0,1.001595
2,Campo Limpo,CAMPO LIMPO,2024,13.4916,7417.0,1.000672
3,Capela Do Socorro,CAPELA DO SOCORRO,2024,14.1123,7092.0,1.000844
4,Casa Verde-Cachoeirinha,CASA VERDE-CACHOEIRINHA,2024,61.9195,3236.0,2.003715
5,Cidade Ademar,CIDADE ADEMAR,2024,21.4087,4673.0,1.000429
6,Cidade Tiradentes,CIDADE TIRADENTES,2024,39.2003,2552.0,1.000392
7,Ermelino Matarazzo,ERMELINO MATARAZZO,2024,48.6145,2057.0,1.000000
8,Freguesia-Brasilândia,FREGUESIA-BRASILANDIA,2024,23.7925,4215.0,1.002854
9,Guaianases,GUAIANASES,2024,56.9638,3511.0,1.999999


In [34]:
df_materna_25 = (
    df_obitos_materna_25
    .loc[:, ['sub.NOME', 'Óbitos_Residentes_MSP']]
    .assign(Ano='2025')
    .merge(
        df_nv[['sub.NOME', 'Ano', 'Número de nascidos vivos']],
        how='left',
        left_on=['sub.NOME', 'Ano'],
        right_on=['sub.NOME', 'Ano']
    )
)
df_materna_25

,sub.NOME,Óbitos_Residentes_MSP,Ano,Número de nascidos vivos
0,ARICANDUVA-FORMOSA-CARRAO,2,2025,2302.0
1,BUTANTA,1,2025,4639.0
2,CAMPO LIMPO,1,2025,7368.0
3,CAPELA DO SOCORRO,3,2025,7121.0
4,CASA VERDE-CACHOEIRINHA,-,2025,3428.0
5,CIDADE ADEMAR,2,2025,4819.0
6,CIDADE TIRADENTES,-,2025,2530.0
7,ERMELINO MATARAZZO,-,2025,2121.0
8,FREGUESIA-BRASILANDIA,-,2025,4293.0
9,GUAIANASES,1,2025,3400.0


In [35]:
df_materna_25['Óbitos_Residentes_MSP'] = df_materna_25['Óbitos_Residentes_MSP'].str.replace('-', '0')
df_materna_25['Óbitos_Residentes_MSP'] = df_materna_25['Óbitos_Residentes_MSP'].astype(float)
df_materna_25

,sub.NOME,Óbitos_Residentes_MSP,Ano,Número de nascidos vivos
0,ARICANDUVA-FORMOSA-CARRAO,2.0,2025,2302.0
1,BUTANTA,1.0,2025,4639.0
2,CAMPO LIMPO,1.0,2025,7368.0
3,CAPELA DO SOCORRO,3.0,2025,7121.0
4,CASA VERDE-CACHOEIRINHA,0.0,2025,3428.0
5,CIDADE ADEMAR,2.0,2025,4819.0
6,CIDADE TIRADENTES,0.0,2025,2530.0
7,ERMELINO MATARAZZO,0.0,2025,2121.0
8,FREGUESIA-BRASILANDIA,0.0,2025,4293.0
9,GUAIANASES,1.0,2025,3400.0


In [36]:
df_materna_25['Razão de mortalidade materna'] = df_materna_25['Óbitos_Residentes_MSP'] / df_materna_25['Número de nascidos vivos'] * 100000
df_materna_25

,sub.NOME,Óbitos_Residentes_MSP,Ano,Número de nascidos vivos,Razão de mortalidade materna
0,ARICANDUVA-FORMOSA-CARRAO,2.0,2025,2302.0,86.880973
1,BUTANTA,1.0,2025,4639.0,21.556370
2,CAMPO LIMPO,1.0,2025,7368.0,13.572204
3,CAPELA DO SOCORRO,3.0,2025,7121.0,42.128914
4,CASA VERDE-CACHOEIRINHA,0.0,2025,3428.0,0.000000
5,CIDADE ADEMAR,2.0,2025,4819.0,41.502386
6,CIDADE TIRADENTES,0.0,2025,2530.0,0.000000
7,ERMELINO MATARAZZO,0.0,2025,2121.0,0.000000
8,FREGUESIA-BRASILANDIA,0.0,2025,4293.0,0.000000
9,GUAIANASES,1.0,2025,3400.0,29.411765


In [37]:
df_materna = pd.concat([df_materna, df_materna_25], ignore_index=True)
df_materna = df_materna.drop(columns=['Subprefeitura'])
df_materna

,sub.NOME,Ano,Razão de mortalidade materna,Número de nascidos vivos,Óbitos_Residentes_MSP
0,ARICANDUVA-FORMOSA-CARRAO,2024,0.000000,2177.0,0.000000
1,BUTANTA,2024,22.768700,4399.0,1.001595
2,CAMPO LIMPO,2024,13.491600,7417.0,1.000672
3,CAPELA DO SOCORRO,2024,14.112300,7092.0,1.000844
4,CASA VERDE-CACHOEIRINHA,2024,61.919500,3236.0,2.003715
...,...,...,...,...,...
60,SE,2025,26.048450,3839.0,1.000000
61,VILA MARIA-VILA GUILHERME,2025,81.081081,3700.0,3.000000
62,VILA MARIANA,2025,0.000000,2721.0,0.000000
63,VILA PRUDENTE,2025,0.000000,2237.0,0.000000
